# Train/Test Classification With Selected Feature Sets

Notebook này đọc `data/all_results_models.xlsx` để lấy classifier và `best_params`, train trực tiếp trên `X_train/y_train`, test trên `X_test/y_test`, chạy lần lượt bản không feature selection và tất cả file trong `data/selected_indices`.

File output chính có đúng các cột: `selected_indices_file`, `classifier`, `accuracy`, `precision`, `recall`, `f1_score`, `mcc`, `cohen_kappa`. Với bài toán multi-class, `precision`, `recall`, và `f1_score` dùng macro average.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running in Google Colab; skipping Drive mount.")

In [ ]:
from pathlib import Path
import os


def resolve_data_path(filename: str) -> str:
    candidates = []

    env_data_dir = os.environ.get("FEATURE_SELECTION_DATA_DIR")
    if env_data_dir:
        candidates.append(Path(env_data_dir) / filename)

    cwd = Path.cwd()
    candidates.extend([
        cwd / "data" / filename,
        cwd.parent / "data" / filename,
        Path("/content/data") / filename,
        Path("/content/feature-selection-experiments/data") / filename,
        Path("/content/drive/MyDrive/feature-selection-experiments/data") / filename,
        Path("/content/drive/MyDrive/data") / filename,
        Path(r"D:\feature-selection-experiments\data") / filename,
    ])

    for path in candidates:
        if path.exists():
            return str(path)

    search_roots = [Path("/content"), Path("/content/drive/MyDrive")]
    for root in search_roots:
        if not root.exists():
            continue
        matches = list(root.glob(f"**/{filename}"))
        for match in matches:
            if match.parent.name == "data":
                return str(match)
        if matches:
            return str(matches[0])

    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Could not find {filename}. Checked:\n{checked}\n\n"
        "Upload data to /content/data, or mount Drive and place it at "
        "/content/drive/MyDrive/feature-selection-experiments/data."
    )


def resolve_selected_indices_dir(data_dir: Path) -> Path:
    candidates = []

    env_selected_dir = os.environ.get("FEATURE_SELECTION_SELECTED_INDICES_DIR")
    if env_selected_dir:
        candidates.append(Path(env_selected_dir))

    cwd = Path.cwd()
    candidates.extend([
        data_dir / "selected_indices",
        cwd / "data" / "selected_indices",
        cwd.parent / "data" / "selected_indices",
        Path("/content/data/selected_indices"),
        Path("/content/feature-selection-experiments/data/selected_indices"),
        Path("/content/drive/MyDrive/feature-selection-experiments/data/selected_indices"),
        Path(r"D:\feature-selection-experiments\data\selected_indices"),
    ])

    for path in candidates:
        if path.exists() and path.is_dir():
            return path

    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Could not find selected_indices folder. Checked:\n{checked}")


DATA_DIR = Path(resolve_data_path("X_train.csv")).parent
SELECTED_INDICES_DIR = resolve_selected_indices_dir(DATA_DIR)

if Path("/content/drive/MyDrive").exists():
    RESULT_DIR = Path("/content/drive/MyDrive/feature-selection-experiments/results/train-test-selected-indices-classification")
else:
    RESULT_DIR = Path.cwd() / "results" / "train-test-selected-indices-classification"
RESULT_DIR.mkdir(parents=True, exist_ok=True)


def output_path(filename: str) -> str:
    return str(RESULT_DIR / filename)


print(f"Using DATA_DIR: {DATA_DIR}")
print(f"Using SELECTED_INDICES_DIR: {SELECTED_INDICES_DIR}")
print(f"Using RESULT_DIR: {RESULT_DIR}")

In [ ]:
import ast
import json
import re
import time
import warnings
import zipfile
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd

from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
X_train = pd.read_csv(resolve_data_path("X_train.csv"))
X_test = pd.read_csv(resolve_data_path("X_test.csv"))

y_train = pd.read_csv(resolve_data_path("y_train.csv")).iloc[:, 0].values
y_test = pd.read_csv(resolve_data_path("y_test.csv")).iloc[:, 0].values

if list(X_train.columns) != list(X_test.columns):
    raise ValueError("X_train and X_test columns do not match.")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)
print("Classes:", np.unique(y_train))

In [ ]:
MODEL_NAME_ALIASES = {
    "KNN": "KNN",
    "DecisionTree": "DecisionTree",
    "RandomForest": "RandomForest",
    "GaussianNB": "GaussianNB",
    "LogisticRegression": "LogisticRegression",
    "LinearSVM": "LinearSVM",
    "RBF_SVM": "RBF_SVM",
}

METHOD_BY_SELECTED_FILE = {
    "bmo_selected_features.csv": "BMO_SVM",
    "gwo_pso_selected_indices.csv": "GWO_PSO_KNN",
    "ibabc_cgo_selected_indices.csv": "iBABC_CGO",
    "pso_selected_indices.csv": "PSO_DT",
    "qlde_spark_diversity_selected_features.csv": "DE_QLearning_Spark_Diversity",
    "sga_selected_indices.csv": "SGA_RF",
}


def _xlsx_col_index(cell_ref: str) -> int:
    match = re.match(r"([A-Z]+)", cell_ref)
    if not match:
        raise ValueError(f"Invalid Excel cell reference: {cell_ref}")
    col = 0
    for ch in match.group(1):
        col = col * 26 + ord(ch) - ord("A") + 1
    return col - 1


def read_xlsx_first_sheet(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    ns = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}

    with zipfile.ZipFile(path) as zf:
        shared_strings = []
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for item in root.findall("a:si", ns):
                shared_strings.append("".join(t.text or "" for t in item.findall(".//a:t", ns)))

        sheet_root = ET.fromstring(zf.read("xl/worksheets/sheet1.xml"))
        rows = []
        for row in sheet_root.findall(".//a:sheetData/a:row", ns):
            values = []
            for cell in row.findall("a:c", ns):
                idx = _xlsx_col_index(cell.attrib["r"])
                cell_type = cell.attrib.get("t")
                value_node = cell.find("a:v", ns)
                inline_node = cell.find("a:is", ns)

                if cell_type == "s" and value_node is not None:
                    value = shared_strings[int(value_node.text)]
                elif cell_type == "inlineStr" and inline_node is not None:
                    value = "".join(t.text or "" for t in inline_node.findall(".//a:t", ns))
                elif value_node is not None:
                    value = value_node.text or ""
                else:
                    value = ""

                while len(values) <= idx:
                    values.append("")
                values[idx] = value
            rows.append(values)

    if not rows:
        return pd.DataFrame()

    width = max(len(row) for row in rows)
    normalized = [row + [""] * (width - len(row)) for row in rows]
    header = normalized[0]
    data = normalized[1:]
    return pd.DataFrame(data, columns=header)


def parse_best_params(value) -> dict:
    if pd.isna(value) or str(value).strip() == "":
        return {}
    parsed = ast.literal_eval(str(value))
    if not isinstance(parsed, dict):
        raise ValueError(f"best_params must parse to dict, got: {value}")
    return parsed


def load_model_best_params(path: str | Path) -> dict[str, dict[str, dict]]:
    df = read_xlsx_first_sheet(path)
    df = df.loc[:, [col for col in df.columns if str(col).strip() != ""]]

    if "Method" not in df.columns or "best_params" not in df.columns:
        raise ValueError("all_results_models.xlsx must contain Method and best_params columns.")

    if "model" in df.columns:
        model_col = "model"
    else:
        model_col = df.columns[0]

    df["Method"] = df["Method"].replace("", np.nan).ffill()
    df = df[
        (df[model_col].astype(str).str.strip() != "")
        & (df["best_params"].astype(str).str.strip() != "")
    ]

    params_by_method = {}
    for _, row in df.iterrows():
        method = str(row["Method"]).strip()
        model_name = MODEL_NAME_ALIASES.get(str(row[model_col]).strip(), str(row[model_col]).strip())
        params_by_method.setdefault(method, {})[model_name] = parse_best_params(row["best_params"])

    return params_by_method


def make_estimator(classifier: str, params: dict):
    params = dict(params)
    if classifier == "KNN":
        return KNeighborsClassifier(**params)
    if classifier == "DecisionTree":
        return DecisionTreeClassifier(random_state=42, **params)
    if classifier == "RandomForest":
        return RandomForestClassifier(random_state=42, n_jobs=-1, **params)
    if classifier == "GaussianNB":
        return GaussianNB(**params)
    if classifier == "LogisticRegression":
        return LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, **params)
    if classifier == "LinearSVM":
        return LinearSVC(max_iter=3000, random_state=42, **params)
    if classifier == "RBF_SVM":
        return SVC(kernel="rbf", random_state=42, **params)
    raise ValueError(f"Unsupported classifier: {classifier}")


ALL_RESULTS_MODELS_PATH = resolve_data_path("all_results_models.xlsx")
PARAMS_BY_METHOD = load_model_best_params(ALL_RESULTS_MODELS_PATH)
DEFAULT_METHOD_FOR_BASELINE = next(iter(PARAMS_BY_METHOD))

print(f"Using ALL_RESULTS_MODELS_PATH: {ALL_RESULTS_MODELS_PATH}")
print(f"Loaded methods: {list(PARAMS_BY_METHOD)}")
print(f"Baseline uses params from method: {DEFAULT_METHOD_FOR_BASELINE}")

In [ ]:
COLUMN_SELECTED_FILES = {
    "bmo_selected_features.csv",
    "qlde_spark_diversity_selected_features.csv",
}


def load_selected_columns(path: Path, all_columns: pd.Index) -> list[str]:
    if path.name in COLUMN_SELECTED_FILES:
        df = pd.read_csv(path)
        if "selected_feature" in df.columns:
            selected = df["selected_feature"].dropna().astype(str).tolist()
        else:
            selected = df.iloc[:, 0].dropna().astype(str).tolist()

        missing = sorted(set(selected) - set(all_columns))
        if missing:
            raise ValueError(
                f"{path.name} contains {len(missing)} feature names not found in X columns. "
                f"Examples: {missing[:10]}"
            )
        return selected

    raw = pd.read_csv(path, header=None).iloc[:, 0].dropna()
    indices = pd.to_numeric(raw, errors="raise").astype(int).tolist()

    invalid = [idx for idx in indices if idx < 0 or idx >= len(all_columns)]
    if invalid:
        raise ValueError(
            f"{path.name} contains {len(invalid)} invalid 0-based column indices. "
            f"Examples: {invalid[:10]}"
        )

    return [all_columns[idx] for idx in indices]


feature_sets = []
for selected_file in sorted(SELECTED_INDICES_DIR.glob("*.csv")):
    selected_columns = load_selected_columns(selected_file, X_train.columns)
    method_name = METHOD_BY_SELECTED_FILE.get(selected_file.name, DEFAULT_METHOD_FOR_BASELINE)
    feature_sets.append((selected_file.name, selected_columns, method_name))
    print(f"{selected_file.name}: {len(selected_columns)} features | params method: {method_name}")

print(f"Total selected feature sets: {len(feature_sets)}")

In [ ]:
def compute_test_metrics(y_true, y_pred) -> dict:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_score": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
    }


METRICS_OUTPUT = output_path("selected_indices_classifier_test_metrics.csv")
USED_PARAMS_OUTPUT = output_path("selected_indices_classifier_used_params.csv")

summary_rows = []
used_params_rows = []

for selected_indices_file, selected_columns, method_name in feature_sets:
    if selected_columns is None:
        X_train_current = X_train
        X_test_current = X_test
    else:
        X_train_current = X_train.loc[:, selected_columns]
        X_test_current = X_test.loc[:, selected_columns]

    if method_name not in PARAMS_BY_METHOD:
        raise ValueError(f"Method {method_name!r} not found in all_results_models.xlsx")

    model_params = PARAMS_BY_METHOD[method_name]
    print(
        f"\n=== Feature set: {selected_indices_file} | "
        f"n_features={X_train_current.shape[1]} | params method={method_name} ==="
    )

    for classifier, params in model_params.items():
        print(f"[{selected_indices_file}] [{classifier}] training with saved best_params...")
        start_time = time.perf_counter()

        estimator = make_estimator(classifier, params)
        estimator.fit(X_train_current, y_train)
        y_pred = estimator.predict(X_test_current)

        elapsed_sec = time.perf_counter() - start_time

        row = {
            "selected_indices_file": selected_indices_file,
            "classifier": classifier,
        }
        row.update(compute_test_metrics(y_test, y_pred))
        summary_rows.append(row)

        used_params_rows.append({
            "selected_indices_file": selected_indices_file,
            "method": method_name,
            "classifier": classifier,
            "n_features": X_train_current.shape[1],
            "best_params": json.dumps(params, ensure_ascii=False),
            "fit_time_sec": elapsed_sec,
        })

        pd.DataFrame(summary_rows).to_csv(METRICS_OUTPUT, index=False, encoding="utf-8-sig")
        pd.DataFrame(used_params_rows).to_csv(USED_PARAMS_OUTPUT, index=False, encoding="utf-8-sig")

        print(
            f"  Test macro-F1: {row['f1_score']:.4f} | "
            f"Accuracy: {row['accuracy']:.4f} | "
            f"Time: {elapsed_sec:.2f}s | "
            f"Params: {params}"
        )

print(f"\nSaved metrics to: {METRICS_OUTPUT}")
print(f"Saved used params to: {USED_PARAMS_OUTPUT}")

In [ ]:
metrics_df = pd.read_csv(METRICS_OUTPUT)
display(metrics_df)
print(f"Rows: {len(metrics_df)}")
print(f"Saved results to: {RESULT_DIR}")